# Preprocessing and partitioning

Recreate the holdout and threshold validation partitions. The split uses present labels solely for stratification. The fitted transform is created inside a model pipeline.

Run the training CLI before executing this notebook.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'configs/full.json').is_file())
sys.path.insert(0, str(root))
from src.utils import read_json

pointer = root / 'models/latest.json'
if not pointer.is_file():
    raise FileNotFoundError('Run python main.py --mode train from the project root first.')
run = root / 'reports/runs' / read_json(pointer)['run_id']
metadata = read_json(run / 'metadata.json')
print(metadata['profile'], metadata['run_id'])

In [ ]:
from src.data_loader import FEATURES, load_dataset
from src.preprocessing import split_data, stratified_subset
from src.utils import DATASET

config = metadata['configuration']
data = load_dataset(DATASET)
pool, holdout, exclusions = split_data(data, config['test_size'], config['seed'])
pool = stratified_subset(pool, config['development_rows'], config['seed'])
fit, threshold, _ = split_data(pool, 0.2, config['seed'] + 1)
assert not set(fit.index).intersection(threshold.index)
assert not set(pool.index).intersection(holdout.index)
display(pd.DataFrame([
    {'partition': name, 'rows': len(part), 'fraud': part.Class.sum()}
    for name, part in [('fit', fit), ('threshold', threshold), ('holdout', holdout)]
]))
print(exclusions)
print(f'Canonical inference schema contains {len(FEATURES)} features.')